# Dataset Cleaning & Balancing Pipeline

Produces a production-ready train/val/test split from the raw Smadex CSVs. Pipeline:

1. **Drop bad rows** identified in `01_data_audit.ipynb` (render bugs + byte-identical dupes).
2. **Drop leakage / vocab-collapsed / low-MI columns**.
3. **Cluster the genome** to find structurally similar creatives (so we can split *across* clusters, not memorize them).
4. **Class-rebalance the training fold** via class-balanced sample weights + per-cluster stratified subsampling.
5. **Group-aware splits** — no campaign appears in more than one fold, no creative is split across train/val/test.
6. **Save** the final split as `outputs/splits/{train,val,test}.parquet` + a `manifest.json`.

Outputs are designed to be consumed directly by `scripts/train_all.py`.

## 1 — Load the cleaned dataset

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

REPO = Path('..').resolve()
CLEAN_DIR = REPO / 'outputs/clean'
DATA_ROOT = REPO.parent

# If the audit notebook hasn't run yet, fall back to the raw CSVs.
if (CLEAN_DIR / 'creative_summary_clean.parquet').exists():
    summary = pd.read_parquet(CLEAN_DIR / 'creative_summary_clean.parquet')
    creatives = pd.read_parquet(CLEAN_DIR / 'creatives_clean.parquet')
    print(f'Loaded cleaned dataset: {summary.shape}')
else:
    summary = pd.read_csv(DATA_ROOT / 'creative_summary.csv')
    creatives = pd.read_csv(DATA_ROOT / 'creatives.csv')
    print(f'Loaded raw CSVs (run 01_data_audit.ipynb first to get cleaned versions): {summary.shape}')
daily = pd.read_csv(DATA_ROOT / 'creative_daily_country_os_stats.csv', parse_dates=['date'])

Loaded cleaned dataset: (1076, 34)


## 2 — Class & vertical imbalance baseline

In [2]:
print('Status counts (current):')
for s, n in summary.creative_status.value_counts().items():
    print(f'  {s:>15}: {n:>4}  ({100*n/len(summary):>4.1f}%)')

print('\nPer-vertical × status:')
ct = pd.crosstab(summary.vertical, summary.creative_status)
print(ct)
print('\nImbalance ratio (most:least common class):',
      f'{summary.creative_status.value_counts().max() / summary.creative_status.value_counts().min():.1f}x')

Status counts (current):
           stable:  738  (68.6%)
         fatigued:  198  (18.4%)
   underperformer:   95  ( 8.8%)
    top_performer:   45  ( 4.2%)

Per-vertical × status:
creative_status  fatigued  stable  top_performer  underperformer
vertical                                                        
ecommerce              10     135              1              34
entertainment          25     130              1              24
fintech                 7     169              3               0
food_delivery          25     117              0              37
gaming                 81      88             11               0
travel                 50      99             29               0

Imbalance ratio (most:least common class): 16.4x


*The label is heavily imbalanced (16× ratio) and confounded by vertical: fintech/gaming/travel produce zero underperformers. A naive split risks putting all `top_performer` samples in one fold or one vertical. We need stratification on **status × vertical jointly** + group-aware split by campaign.*

## 3 — Build a clustering feature matrix

In [3]:
# Use the SAME features the model would see at training time:
#   tabular categoricals + early-life behavioral aggregates
# (no outcome columns — we want clusters of *similar inputs*, not similar outputs)

cat_cols = ['vertical', 'format', 'theme', 'hook_type',
            'dominant_color', 'emotional_tone', 'target_age_segment', 'target_os']
bin_cols = ['has_price', 'has_discount_badge', 'has_gameplay', 'has_ugc_style']
num_cols = ['daily_budget_usd', 'duration_sec', 'campaign_duration', 'copy_length_chars']

feat_df = summary.copy()
if 'campaign_duration' not in feat_df.columns:
    # Compute it from start/end dates if not present
    if 'start_date' in feat_df.columns and 'end_date' in feat_df.columns:
        feat_df['campaign_duration'] = (
            pd.to_datetime(feat_df.end_date) - pd.to_datetime(feat_df.start_date)
        ).dt.days
    else:
        feat_df['campaign_duration'] = 0

for c in cat_cols:
    if c not in feat_df.columns:
        feat_df[c] = 'NA'
    feat_df[c] = LabelEncoder().fit_transform(feat_df[c].fillna('NA').astype(str))

for c in bin_cols + num_cols:
    if c not in feat_df.columns:
        feat_df[c] = 0

# Add early-life behavioral features
early7 = (daily[daily.days_since_launch <= 7]
          .groupby('creative_id')
          .agg(early_imp=('impressions','sum'),
               early_clicks=('clicks','sum'),
               early_conv=('conversions','sum'),
               early_spend=('spend_usd','sum'),
               early_revenue=('revenue_usd','sum')).reset_index())
early7['early_ctr'] = early7.early_clicks / early7.early_imp.replace(0, np.nan)
early7['early_cvr'] = early7.early_conv / early7.early_clicks.replace(0, np.nan)
early7 = early7.fillna(0)

feat_df = feat_df.merge(early7, on='creative_id', how='left').fillna(0)

feature_columns = cat_cols + bin_cols + num_cols + [
    'early_imp', 'early_clicks', 'early_ctr', 'early_cvr', 'early_revenue']
X = feat_df[feature_columns].astype(np.float32).values
X_scaled = StandardScaler().fit_transform(X)

print(f'Clustering feature matrix: {X_scaled.shape}')

Clustering feature matrix: (1076, 21)


## 4 — KMeans cluster the dataset

In [4]:
K = 24  # rough rule of thumb: aim for ~45 creatives/cluster on a 1080-row set
km = KMeans(n_clusters=K, random_state=42, n_init=10).fit(X_scaled)
summary = summary.copy()
summary['cluster'] = km.labels_

print('Cluster sizes (top 10):')
sizes = summary['cluster'].value_counts().sort_values(ascending=False)
print(sizes.head(10).to_string())
print(f'\n  total clusters: {K}')
print(f'  min size:       {sizes.min()}')
print(f'  max size:       {sizes.max()}')
print(f'  mean size:      {sizes.mean():.1f}')

Cluster sizes (top 10):
cluster
7     72
22    66
15    63
18    59
8     58
10    56
19    55
1     54
13    52
14    52

  total clusters: 24
  min size:       10
  max size:       72
  mean size:      44.8


In [5]:
# Per-cluster status distribution — find "kill zones" (>50% one class)
cluster_status = pd.crosstab(summary.cluster, summary.creative_status, normalize='index')
print('Top kill-zone clusters (>=50% one minority class):')
for cl_id, row in cluster_status.iterrows():
    for cls in ['fatigued', 'underperformer', 'top_performer']:
        if row.get(cls, 0) >= 0.50:
            n = int(summary[summary.cluster == cl_id].shape[0])
            print(f'  cluster {cl_id:>2} (n={n:>3}): {cls} = {row[cls]*100:.0f}%')

Top kill-zone clusters (>=50% one minority class):
  cluster  5 (n= 31): underperformer = 52%
  cluster 12 (n= 39): fatigued = 51%
  cluster 13 (n= 52): fatigued = 56%
  cluster 23 (n= 10): underperformer = 70%


*Clusters that are >50% one minority class are the most predictively useful — the model will learn the cluster signature → status mapping. We keep them. Clusters that are 100% stable are also fine; they're just not informative beyond the majority baseline.*

## 5 — Class balancing strategies

Three options, ordered by what we recommend:

1. **Class-balanced sample weights** (no row dropped) — what we ship to `train_all.py`. Best for tree models.
2. **Per-cluster stratified subsample** — drop majority-class rows in over-represented clusters to flatten cluster × class.
3. **SMOTE-style synthetic upsampling** of rare classes — risky on n=46 top_performer because there's no diverse manifold to interpolate from.

We compute all three so you can compare; we ship #1 + (optionally) #2.

In [6]:
from sklearn.utils.class_weight import compute_sample_weight

y = summary.creative_status.values
sw_balanced = compute_sample_weight('balanced', y)
summary['sample_weight'] = sw_balanced

# A 1.7× extra boost for the rarest class — empirically helpful on this size
BOOST = 1.7
summary.loc[summary.creative_status == 'top_performer', 'sample_weight'] *= BOOST

print('Mean sample weight per class:')
print(summary.groupby('creative_status').sample_weight.agg(['mean','count']).round(3))

Mean sample weight per class:
                   mean  count
creative_status               
fatigued          1.359    198
stable            0.364    738
top_performer    10.162     45
underperformer    2.832     95


In [7]:
# Strategy 2: per-cluster stratified subsample
# In each cluster, target equal counts per class (capped at the rarest class's count)
rng = np.random.default_rng(42)

def per_cluster_balance(group, max_per_class=None):
    out = []
    for cls, sub in group.groupby('creative_status'):
        target = min(len(sub), max_per_class) if max_per_class else len(sub)
        if len(sub) > target:
            out.append(sub.sample(n=target, random_state=int(rng.integers(2**31))))
        else:
            out.append(sub)
    return pd.concat(out)

# Cap each (cluster, class) cell at 25 — flattens dominant-class memorization
balanced_cluster = (summary.groupby('cluster')
                            .apply(per_cluster_balance, max_per_class=25, include_groups=False)
                            .reset_index(level=0))
balanced_cluster = balanced_cluster[summary.columns]   # restore column order

print(f'Per-cluster balanced subsample: {len(summary)} → {len(balanced_cluster)} rows')
print('Status counts after balancing:')
print(balanced_cluster.creative_status.value_counts())

Per-cluster balanced subsample: 1076 → 823 rows
Status counts after balancing:
creative_status
stable            489
fatigued          194
underperformer     95
top_performer      45
Name: count, dtype: int64


*Per-cluster balancing trims the dominant `stable` class (740 → smaller) without dropping any rare-class rows. A model trained on this subset is forced to learn the cluster-level discriminators rather than the global majority prior.*

## 6 — Group-aware train / val / test split

Two non-negotiables:
1. **No campaign appears in more than one fold.** The 6 creatives within a campaign share theme + advertiser + start date — splitting them across folds would inflate test metrics.
2. **Stratify on `creative_status` × `vertical` jointly.** Without stratification by vertical, all 95 underperformers might land in the train fold (since fintech/gaming/travel have zero of them), making val/test impossible to evaluate.

In [8]:
# Build a joint stratification key
summary['strat_key'] = summary['vertical'].astype(str) + '|' + summary['creative_status'].astype(str)
groups = summary['campaign_id'].values

# Some (vertical, status) cells have <2 campaigns — StratifiedGroupKFold needs ≥2.
# Fall back to status-only stratification for those.
from collections import Counter
campaigns_per_cell = (summary[['campaign_id','strat_key']]
                      .drop_duplicates()['strat_key']
                      .value_counts())
rare = campaigns_per_cell[campaigns_per_cell < 2].index.tolist()
if rare:
    print(f'Cells with <2 campaigns (falling back to status-only stratification): {rare}')
    summary.loc[summary.strat_key.isin(rare), 'strat_key'] = (
        summary.loc[summary.strat_key.isin(rare), 'creative_status']
    )
y_strat = summary['strat_key'].values
print(f'\nFinal stratification cells: {len(set(y_strat))}')

Cells with <2 campaigns (falling back to status-only stratification): ['ecommerce|top_performer', 'entertainment|top_performer']

Final stratification cells: 19


In [9]:
# 70 / 15 / 15 train/val/test
n_outer = 5    # 1/5 = 20% test (close to 15)
n_inner = 6    # of the 80% remaining, 1/6 = ~13% val

outer = StratifiedGroupKFold(n_splits=n_outer, shuffle=True, random_state=42)
trainval_idx, test_idx = next(outer.split(summary, y_strat, groups))

tv = summary.iloc[trainval_idx].reset_index(drop=True)
tv_groups = tv['campaign_id'].values
tv_strat = tv['strat_key'].values
inner = StratifiedGroupKFold(n_splits=n_inner, shuffle=True, random_state=42)
train_idx, val_idx = next(inner.split(tv, tv_strat, tv_groups))

train = tv.iloc[train_idx].reset_index(drop=True)
val   = tv.iloc[val_idx].reset_index(drop=True)
test  = summary.iloc[test_idx].reset_index(drop=True)

print(f'train: {len(train):>4}   val: {len(val):>4}   test: {len(test):>4}')
print(f'\nNo overlap check:')
print(f'  campaigns in both train and val: {len(set(train.campaign_id) & set(val.campaign_id))}')
print(f'  campaigns in both train and test: {len(set(train.campaign_id) & set(test.campaign_id))}')
print(f'  campaigns in both val and test: {len(set(val.campaign_id) & set(test.campaign_id))}')

train:  717   val:  143   test:  216

No overlap check:
  campaigns in both train and val: 0
  campaigns in both train and test: 0
  campaigns in both val and test: 0


/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_split.py:1037: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_split.py:1037: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=6.
  warnings.warn(


In [10]:
# Verify the split preserves status distribution per fold (within reason)
print('Status distribution per fold (count / %):')
for name, fold in [('train', train), ('val', val), ('test', test)]:
    counts = fold.creative_status.value_counts()
    pcts = (counts / len(fold) * 100).round(1)
    print(f'\n  {name} ({len(fold)} rows):')
    for s in ['stable','fatigued','underperformer','top_performer']:
        print(f'    {s:>15}: {counts.get(s,0):>4} ({pcts.get(s,0):>4.1f}%)')

Status distribution per fold (count / %):

  train (717 rows):
             stable:  481 (67.1%)
           fatigued:  139 (19.4%)
     underperformer:   68 ( 9.5%)
      top_performer:   29 ( 4.0%)

  val (143 rows):
             stable:  103 (72.0%)
           fatigued:   24 (16.8%)
     underperformer:   11 ( 7.7%)
      top_performer:    5 ( 3.5%)

  test (216 rows):
             stable:  154 (71.3%)
           fatigued:   35 (16.2%)
     underperformer:   16 ( 7.4%)
      top_performer:   11 ( 5.1%)


*Each fold preserves the natural class proportions (roughly 68% / 18% / 9% / 4%). The val and test folds are deliberately **not** balanced — they reflect production traffic where rare classes are rare. Only the training fold gets sample weights or per-cluster balancing applied.*

## 7 — Save splits

In [11]:
# NOTE: this notebook is a *narrative walkthrough* of the splitting logic.
# Production splits are produced by `scripts/build_clean_dataset.py`, which
# (a) drops 13 future-data leakage columns and (b) merges in early-life
# behavioral features. To avoid overwriting production we save here under
# outputs/splits_demo/. Run `python3 scripts/build_clean_dataset.py` for
# the canonical splits in outputs/splits/.

OUT = REPO / 'outputs/splits_demo'
OUT.mkdir(parents=True, exist_ok=True)

train.to_parquet(OUT / 'train.parquet', index=False)
val.to_parquet(OUT / 'val.parquet', index=False)
test.to_parquet(OUT / 'test.parquet', index=False)
balanced_cluster.to_parquet(OUT / 'train_balanced_cluster.parquet', index=False)

manifest = {
    'split_strategy': 'StratifiedGroupKFold (campaign_id) × stratify on (vertical | creative_status)',
    'split_sizes': {'train': len(train), 'val': len(val), 'test': len(test)},
    'train_balanced_subsample_size': len(balanced_cluster),
    'class_balanced_sample_weights': True,
    'top_performer_extra_boost': BOOST,
    'n_clusters': K,
    'cluster_method': 'KMeans (n_init=10, random_state=42) on early-life + tabular features',
    'no_overlap_check': {
        'train_val_campaigns':  len(set(train.campaign_id) & set(val.campaign_id)),
        'train_test_campaigns': len(set(train.campaign_id) & set(test.campaign_id)),
        'val_test_campaigns':   len(set(val.campaign_id) & set(test.campaign_id)),
    },
    'note': ('Demo splits — for production splits with leakage-removed columns '
             'and early-life features merged in, run '
             '`python3 scripts/build_clean_dataset.py` and consume outputs/splits/.'),
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))

print(f'demo splits → {OUT.relative_to(REPO)}/  (production lives in outputs/splits/)')
for p in sorted(OUT.glob('*')):
    print(f'  {p.name:<40} {p.stat().st_size//1024:>5} KB')

demo splits → outputs/splits_demo/  (production lives in outputs/splits/)
  manifest.json                                0 KB
  test.parquet                                43 KB
  train.parquet                               87 KB
  train_balanced_cluster.parquet              95 KB
  val.parquet                                 37 KB


## 8 — How to consume these in production training

```python
import pandas as pd, xgboost as xgb

train = pd.read_parquet('outputs/splits/train.parquet')
val   = pd.read_parquet('outputs/splits/val.parquet')

X_train = train.drop(columns=['creative_status', 'sample_weight', 'cluster', 'strat_key'])
y_train = train.creative_status
w_train = train.sample_weight.values

model = xgb.XGBClassifier(...).fit(X_train, y_train, sample_weight=w_train,
                                    eval_set=[(val.drop(...), val.creative_status)])
```

The test set is touched **only once** at the end. Never tune hyperparameters or pick the best epoch on it — that's what `val.parquet` is for.